In [ ]:
# Celda 1 — imports
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, SILVER_DB, GOLD_DB

In [ ]:
# Celda 2 — función genérica (misma lógica)
def cargar_a_gold(nombre_tabla, query):
    print(f"Leyendo silver para {nombre_tabla}...")
    rows, cols = CH.execute(query, with_column_types=True)
    df = pd.DataFrame(rows, columns=[c[0] for c in cols])

    df['_valid_from'] = datetime.now()
    df = df.where(pd.notnull(df), None)

    CH.execute(f"TRUNCATE TABLE {GOLD_DB}.{nombre_tabla}")
    CH.execute(f"INSERT INTO {GOLD_DB}.{nombre_tabla} VALUES", df.to_dict('records'))
    print(f"✓ {len(df):,} filas cargadas en gold.{nombre_tabla}")

In [26]:
# Celda 3 — definición de tablas y queries
tablas = {
    'dim_customers': f"""
        SELECT
            customer_id,
            company_name,
            contact_name,
            contact_title,
            city,
            region,
            country,
            phone
        FROM {SILVER_DB}.stg_customers
    """,

    'dim_employees': f"""
        SELECT
            employee_id,
            full_name,
            title,
            title_of_courtesy,
            birth_date,
            hire_date,
            city,
            region,
            country
        FROM {SILVER_DB}.stg_employees
    """,

    'dim_products': f"""
        SELECT
            p.product_id,
            p.product_name,
            p.category_id,
            c.category_name,
            p.supplier_id,
            s.company_name  AS supplier_name,
            s.country       AS supplier_country,
            p.quantity_per_unit,
            p.unit_price    AS list_price,
            p.discontinued
        FROM {SILVER_DB}.stg_products p
        LEFT JOIN {SILVER_DB}.stg_categories c ON p.category_id = c.category_id
        LEFT JOIN {SILVER_DB}.stg_suppliers  s ON p.supplier_id = s.supplier_id
    """,

    'dim_shippers': f"""
        SELECT
            shipper_id,
            company_name,
            phone
        FROM {SILVER_DB}.stg_shippers
    """,

    'dim_territories': f"""
        SELECT
            t.territory_id,
            t.territory_description,
            t.region_id,
            r.region_description
        FROM {SILVER_DB}.stg_territories t
        LEFT JOIN {SILVER_DB}.stg_region r ON t.region_id = r.region_id
    """,

    'fact_sales': f"""
    SELECT
        cityHash64(od.order_id, od.product_id)           AS sale_id,
        od.order_id,
        toInt32(formatDateTime(o.order_date, '%Y%m%d'))  AS date_key,
        o.customer_id,
        o.employee_id,
        od.product_id,
        o.ship_via                                        AS shipper_id,
        od.unit_price                                     AS sale_price,
        od.quantity,
        od.discount,
        round(od.unit_price * od.quantity * (1 - od.discount), 4) AS line_total,
        o.freight
    FROM {SILVER_DB}.stg_order_details od
    LEFT JOIN {SILVER_DB}.stg_orders o ON od.order_id = o.order_id """,
}

In [27]:
# Reemplazar la query de dim_date en el diccionario
tablas['dim_date'] = None  # no usa query SQL

# Y agregar una función especial para dim_date
def cargar_dim_date():
    print("Generando dim_date...")
    dates = pd.date_range('1990-01-01', '2030-12-31', freq='D')

    df = pd.DataFrame({
        'date_key':     dates.strftime('%Y%m%d').astype(int),
        'full_date':    dates.date,
        'year':         dates.year.astype(int),
        'quarter':      dates.quarter.astype(int),
        'quarter_name': ('Q' + dates.quarter.astype(str)),
        'month':        dates.month.astype(int),
        'month_name':   dates.strftime('%B'),
        'week':         dates.isocalendar().week.astype(int),
        'day':          dates.day.astype(int),
        'day_name':     dates.strftime('%A'),
        'is_weekend':   (dates.weekday >= 5).astype(int),
    })

    CH.execute(f"TRUNCATE TABLE {GOLD_DB}.dim_date")
    CH.execute(
        f"INSERT INTO {GOLD_DB}.dim_date ({', '.join(df.columns)}) VALUES",
        df.to_dict('records')
    )
    print(f"✓ {len(df):,} filas cargadas en gold.dim_date")

In [28]:
cargar_dim_date()

Generando dim_date...
✓ 14,975 filas cargadas en gold.dim_date


In [29]:
# Celda 4 — ejecutar dims primero, fact al final
dims  = ['dim_customers','dim_employees','dim_products',
         'dim_shippers','dim_territories']
facts = ['fact_sales']

for tabla in dims + facts:
    cargar_a_gold(tabla, tablas[tabla])

Leyendo silver para dim_customers...
✓ 91 filas cargadas en gold.dim_customers
Leyendo silver para dim_employees...
✓ 9 filas cargadas en gold.dim_employees
Leyendo silver para dim_products...
✓ 77 filas cargadas en gold.dim_products
Leyendo silver para dim_shippers...
✓ 6 filas cargadas en gold.dim_shippers
Leyendo silver para dim_territories...
✓ 53 filas cargadas en gold.dim_territories
Leyendo silver para fact_sales...
✓ 2,155 filas cargadas en gold.fact_sales
